In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import Dataset

ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from imdb_spoiler_io import load_raw_imdb_spoiler_json, prepare_reviews_dataframe
from splitters import SplitConfig, split_by_movie_id
from subsample import stratified_sample_by_label
from head_tail import apply_head_tail_truncation
from sampling_utils import create_balanced_splits_by_movie

paths = get_paths(ROOT) 

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_raw = load_raw_imdb_spoiler_json(paths.data_raw)
df_all, _ = prepare_reviews_dataframe(df_raw)
print(f"Starting Dataset Length: {len(df_all)}")

Starting Dataset Length: 573913


In [3]:
print("\nGenerazione Split Strategici (Movie-Aware)...")
train_df, val_df, test_df = create_balanced_splits_by_movie(
    df_all,
    movie_col="movie_id",  # Assicurati che questa colonna esista nel df_all!
    train_total_size=100_000, 
    val_test_size=5_000,
    seed=42
)


Generazione Split Strategici (Movie-Aware)...
--- MOVIE SPLIT REPORT ---
Movies Totali: 1572
Pool Train: 461095 reviews (da 1257 film)
Pool Val:   54284 reviews (da 157 film)
Pool Test:  58534 reviews (da 158 film)

--- BALANCING TRAIN ---
Spoilers disponibili nel Train Pool: 121445
Target richiesto per classe: 50000
Effettivo per classe: 50000


In [4]:
# --- VERIFICA RIGOROSA DEL BILANCIAMENTO ---
def audit_balance(df, split_name):
    total = len(df)
    n_pos = df['label'].sum()
    n_neg = total - n_pos
    ratio_pos = n_pos / total
    
    print(f"📊 AUDIT {split_name.upper()}")
    print(f"   - Totale righe: {total}")
    print(f"   - Spoilers (1): {n_pos} ({ratio_pos:.2%})")
    print(f"   - Normali (0):  {n_neg} ({1-ratio_pos:.2%})")
    
    # Assertions per bloccare l'esecuzione se qualcosa non va
    if split_name == "TRAIN":
        # Tolleranza minima per numeri dispari
        assert 0.49 <= ratio_pos <= 0.51, f"❌ ERRORE CRITICO: Il Train non è bilanciato! Trovato {ratio_pos:.2%}"
        print("   ✅ STATUS: PERFETTAMENTE BILANCIATO (Target 50%)")
    else:
        # Val e Test devono essere sbilanciati (tra il 20% e il 30% di solito per questo dataset)
        if ratio_pos > 0.4:
            print("   ⚠️ WARNING: Val/Test sembrano troppo bilanciati. Sicuro di volerli così?")
        else:
            print("   ✅ STATUS: DISTRIBUZIONE NATURALE (Sbilanciata)")
    print("-" * 30)

audit_balance(train_df, "TRAIN")
audit_balance(val_df, "VAL")
audit_balance(test_df, "TEST")

📊 AUDIT TRAIN
   - Totale righe: 100000
   - Spoilers (1): 50000 (50.00%)
   - Normali (0):  50000 (50.00%)
   ✅ STATUS: PERFETTAMENTE BILANCIATO (Target 50%)
------------------------------
📊 AUDIT VAL
   - Totale righe: 5000
   - Spoilers (1): 1281 (25.62%)
   - Normali (0):  3719 (74.38%)
   ✅ STATUS: DISTRIBUZIONE NATURALE (Sbilanciata)
------------------------------
📊 AUDIT TEST
   - Totale righe: 5000
   - Spoilers (1): 1275 (25.50%)
   - Normali (0):  3725 (74.50%)
   ✅ STATUS: DISTRIBUZIONE NATURALE (Sbilanciata)
------------------------------


In [5]:
def print_dist(df, name):
    pos = df['label'].sum()
    tot = len(df)
    print(f"[{name}] Tot: {tot} | Spoiler: {pos} ({pos/tot:.2%})")

print_dist(train_df, "TRAIN (Balanced 50/50)")
print_dist(val_df,   "VAL   (Real ~26%)")
print_dist(test_df,  "TEST  (Real ~26%)")

[TRAIN (Balanced 50/50)] Tot: 100000 | Spoiler: 50000 (50.00%)
[VAL   (Real ~26%)] Tot: 5000 | Spoiler: 1281 (25.62%)
[TEST  (Real ~26%)] Tot: 5000 | Spoiler: 1275 (25.50%)


In [6]:
MODEL_NAME = "bert-base-uncased"
print("\nApplicazione Head+Tail...")

# Questo passaggio impiegherà circa 10-15 minuti sulla tua CPU per 110k righe totali
train_ht = apply_head_tail_truncation(train_df, MODEL_NAME)
val_ht = apply_head_tail_truncation(val_df, MODEL_NAME)
test_ht = apply_head_tail_truncation(test_df, MODEL_NAME)


Applicazione Head+Tail...
Head-tail strategy: Head=128, Tail=382, total= 512


Token indices sequence length is longer than the specified maximum sequence length for this model (653 > 512). Running this sequence through the model will result in indexing errors
Applying Head+Tail: 100%|██████████| 100000/100000 [00:00<00:00, 183469.36it/s]


Head-tail strategy: Head=128, Tail=382, total= 512


Token indices sequence length is longer than the specified maximum sequence length for this model (997 > 512). Running this sequence through the model will result in indexing errors
Applying Head+Tail: 100%|██████████| 5000/5000 [00:00<00:00, 217391.29it/s]


Head-tail strategy: Head=128, Tail=382, total= 512


Token indices sequence length is longer than the specified maximum sequence length for this model (650 > 512). Running this sequence through the model will result in indexing errors
Applying Head+Tail: 100%|██████████| 5000/5000 [00:00<00:00, 199978.26it/s]


In [7]:
# --- VERIFICA VISIVA HEAD+TAIL ---
from transformers import AutoTokenizer
import random

# Ricarichiamo il tokenizer solo per decodificare e leggere
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def audit_head_tail_content(df_processed, df_original, num_samples=3):
    print(f"\n🔍 AUDIT HEAD+TAIL STRATEGY ({num_samples} esempi casuali)")
    
    # Filtriamo solo review molto lunghe nell'originale per vedere l'effetto del taglio
    # Cerchiamo indici dove il testo originale è > 4000 caratteri (così siamo sicuri che venga tagliato)
    long_reviews_idx = df_original[df_original['text'].str.len() > 4000].index
    
    # Se non ce ne sono abbastanza, prendiamo indici a caso
    if len(long_reviews_idx) < num_samples:
        sample_indices = random.sample(range(len(df_processed)), num_samples)
    else:
        sample_indices = random.sample(list(long_reviews_idx), num_samples)

    for idx in sample_indices:
        # Recuperiamo i dati
        input_ids = df_processed.iloc[idx]['input_ids']
        original_text = df_original.iloc[idx]['text'] # Assumendo che gli indici siano allineati reset_index
        
        # Decodifichiamo i token
        decoded_text = tokenizer.decode(input_ids)
        
        print(f"\n📝 REVIEW ID: {idx}")
        print(f"   Lunghezza Token: {len(input_ids)} (Deve essere <= 512)")
        
        # Check lunghezza
        if len(input_ids) == 512:
            print("   ✅ Lunghezza satura a 512 token.")
        else:
            print(f"   ℹ️ Lunghezza inferiore al max ({len(input_ids)}), probabilmente non è stata tagliata.")

        # Visualizziamo l'inizio e la fine decodificati
        print("   --- INIZIO DECODIFICATO (HEAD) ---")
        print("   " + decoded_text[:300] + " ...")
        
        print("   --- FINE DECODIFICATA (TAIL) ---")
        print("   ... " + decoded_text[-300:])
        
        # VERIFICA DEL "SALTO"
        # Controlliamo se la parte centrale del testo originale MANCA nel testo decodificato
        middle_index = len(original_text) // 2
        middle_snippet = original_text[middle_index : middle_index+50]
        
        # Pulizia base per il check (lowercase)
        if middle_snippet.lower() not in decoded_text.lower() and len(input_ids) == 512:
             print("   ✅ SUCCESSO: Il centro del testo originale è stato rimosso (Head+Tail attiva).")
        elif len(input_ids) < 512:
             print("   ℹ️ INFO: Testo breve, nessun taglio necessario.")
        else:
             print("   ⚠️ WARNING: Potrei aver trovato il centro del testo. Controlla manualmente!")

# Eseguiamo il check sul Train Set (che contiene le Head+Tail applicate)
# NOTA: Assicurati che 'train_df' (originale con testo) e 'train_ht' (processato) siano allineati
# Se train_ht è stato creato da train_df senza shuffle nel mezzo, gli indici corrispondono.
audit_head_tail_content(train_ht, train_df)


🔍 AUDIT HEAD+TAIL STRATEGY (3 esempi casuali)

📝 REVIEW ID: 97292
   Lunghezza Token: 512 (Deve essere <= 512)
   ✅ Lunghezza satura a 512 token.
   --- INIZIO DECODIFICATO (HEAD) ---
   [CLS] michael cera ( superbad, juno ) is scott pilgrim, an unemployed 22 - year - old who plays bass guitar in a rock band known as the sex bob - omb in edgar wright ' s zany comedy scott pilgrim vs. the world. scott is a scrawny kid who looks and acts like sixteen, yet he is a kung fu master with s ...
   --- FINE DECODIFICATA (TAIL) ---
   ...  yes, this wacky little film is not designed to be taken seriously any more than you would consider snow white as a representation of the house - bound submissive female, or tarzan, a white child raised by mangani great apes, as a completely well - adjusted adult without any discernible flaws. [SEP]
   ✅ SUCCESSO: Il centro del testo originale è stato rimosso (Head+Tail attiva).

📝 REVIEW ID: 84133
   Lunghezza Token: 512 (Deve essere <= 512)
   ✅ Lunghezza sa

In [8]:
train_ht.to_parquet(paths.data_processed / "train_100k_balanced.parquet", index=False)
val_ht.to_parquet(paths.data_processed / "val_real.parquet", index=False)
test_ht.to_parquet(paths.data_processed / "test_real.parquet", index=False)